# Phase 6 (post hoc): Huang's stage and the AF veto chosen on 400 Icentia11k patients (phase6_rule_tuning, P20)

PREREGISTRATION §10 v19, P20. Per condition, one of 50 settings (the ten-run ensemble alone; Huang et al.'s stage at seven
thresholds with no veto or P16's veto at six thresholds) is chosen by S F1 on P14's 400 Icentia11k patients, which none of
P18's sixty runs has seen, and written to `choice.json`; P19's patients (rebuilt, hash-checked) then compare it with P19's
setting and with the ensemble alone. One DS2 read compares it with P18's setting.

| `PART` | accelerator | inputs | what it does |
|---|---|---|---|
| `build` | CPU | `ecgaug-repo` | downloads and builds both patient sets (all CPU cores) into `/kaggle/working/rule_tuning` |
| `run` | GPU | `ecgaug-repo`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs`, the `build` session's output | predicts both sets with the 60 runs, chooses on the tuning patients, scores P19's patients |
| `score` | CPU | as `run`, plus the `run` session's output | the choice and the scores again from the saved probabilities |
| `test` | CPU | `ecgaug-repo`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs` | the one posthoc DS2 read (logged) |

`run` and `score` need `results/phase6_rule_tuning/data.json` from `build` in the repository; `test` needs `choice.json`.
Settings: **Internet on** (PhysioNet's open-data mirror). Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`.

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
PART = "build"                        # "build" (CPU), "run" (GPU), "score" (CPU) or "test" (CPU, the DS2 read)
BEATS_DIR = "/kaggle/working/rule_tuning"  # build writes here; run and score search /kaggle/input for the build output
RUN_TESTS = False                     # the full test suite first; nothing else runs if it fails
INPUTS_DIR = "/kaggle/input"          # attached datasets: banks, checkpoints (runs/), external data (data/)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_rule_tuning/rule_tuning.yaml"]
beats = BEATS_DIR if PART == "build" else INPUTS_DIR
extra = ["--part", PART, "--device", DEVICE, "--beats-dir", beats]
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_rule_tuning", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", INPUTS_DIR]
if PART in ("build", "test"):
    cmd += ["--data-dir", INPUTS_DIR]   # not for run and score: the build output's repo/ folder holds a data/ folder too
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
if PART == "test":
    env["ALLOW_TEST_POSTHOC"] = "1"   # the one posthoc DS2 read of PREREGISTRATION v19; it is logged in docs/TEST_ACCESS_LOG.md
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
